# SYMCA Attendance Prediction — 03 Model Training & Evaluation (FINAL)

**Input:** `symca_cleaned.csv` produced by `SYMCA_02_Feature_Engineering_FINAL.ipynb`

**Target:** `Attendence Percentage`

This notebook:
1. Loads the cleaned/engineered dataset.
2. Removes target leakage and post-lecture information.
3. Uses a chronological 80/20 train-test split.
4. Builds preprocessing inside a Scikit-learn Pipeline.
5. Compares four regression algorithms.
6. Evaluates MAE, RMSE, MAPE and R².
7. Selects the best model based on the test-set results.
8. Retrains the selected pipeline on all available cleaned data.
9. Saves the experiment table and deployment-ready `.pkl` model.

**Important:** Do not use `Students Present` as a model input because it is used to calculate the target attendance percentage and is only known after attendance is recorded.


## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("Libraries imported successfully.")


## 2. Find `symca_cleaned.csv` in Kaggle

Run this cell first and copy the exact path shown for `symca_cleaned.csv`.

If Notebook 2 created the file in `/kaggle/working`, either attach that output as a Kaggle Dataset or upload the cleaned CSV to a Kaggle Dataset before running this notebook.


In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))


## 3. Load the Cleaned Dataset

In [ ]:
# Replace this with the exact Kaggle path printed above.
# Example:
# DATA_PATH = "/kaggle/input/symca-attendance-cleaned/symca_cleaned.csv"

DATA_PATH = "/kaggle/input/YOUR-CLEANED-DATASET-NAME/symca_cleaned.csv"

df = pd.read_csv(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values(["Date", "Section", "Start Time Minutes", "Lecture_No"]).reset_index(drop=True)

print("Shape:", df.shape)
display(df.head())


## 4. Define Target and Remove Leakage / Non-Predictive Columns

The model predicts attendance percentage for a lecture.

The following are excluded:
- `Attendence Percentage`: target itself.
- `Students Present`: directly determines the target and is known only after the lecture.
- `Date`: replaced by engineered temporal variables.
- `Start_Time`, `End_Time`: raw text versions of time; engineered time features are used instead.

Constant columns are also removed automatically because they contain no predictive variation.


In [ ]:
target = "Attendence Percentage"

leakage_cols = [
    "Students Present",
    target,
    "Date",
    "Start_Time",
    "End_Time"
]

constant_cols = [
    c for c in df.columns
    if c not in leakage_cols and df[c].nunique(dropna=False) <= 1
]

drop_cols = list(dict.fromkeys(leakage_cols + constant_cols))

X = df.drop(columns=drop_cols)
y = df[target].astype(float)

print("Dropped leakage/non-model columns:")
print(leakage_cols)

print("\nConstant columns dropped:")
print(constant_cols)

print("\nFeatures entering the model:")
print(X.columns.tolist())
print("\nNumber of features before encoding:", X.shape[1])


## 5. Remove Redundant Engineered Features

`Week Number` is a deterministic bucket of `Day of Semester`, so both provide almost the same information.

We keep `Day of Semester` and remove `Week Number` to reduce redundancy.


In [ ]:
if "Week Number" in X.columns:
    X = X.drop(columns=["Week Number"])

print("Final raw feature count:", X.shape[1])
print(X.columns.tolist())


## 6. Check Numeric Multicollinearity

This is an EDA/model-stability diagnostic, not the basis for deleting features automatically.

The final decision here is only to remove the obvious deterministic `Week Number` redundancy.


In [ ]:
numeric_for_corr = X.select_dtypes(include=np.number)

if numeric_for_corr.shape[1] >= 2:
    corr_matrix = numeric_for_corr.corr().abs()

    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            value = corr_matrix.iloc[i, j]
            if pd.notna(value) and value > 0.90:
                high_corr_pairs.append(
                    (
                        corr_matrix.columns[i],
                        corr_matrix.columns[j],
                        round(value, 3)
                    )
                )

    print("Highly correlated numeric pairs (>|0.90|):")
    if high_corr_pairs:
        for a, b, c in high_corr_pairs:
            print(f"{a} <-> {b}: {c}")
    else:
        print("None found.")
else:
    print("Not enough numeric features for a correlation-pair check.")


## 7. Chronological Train/Test Split

Because this project predicts future attendance, a random split is not used.

The first 80% of observations by date are used for training and the latest 20% are held out for testing.


In [ ]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training period:",
      df.loc[0, "Date"].date(),
      "to",
      df.loc[split_index - 1, "Date"].date())

print("Testing period:",
      df.loc[split_index, "Date"].date(),
      "to",
      df.loc[len(df) - 1, "Date"].date())


## 8. Build the Preprocessing Pipeline

All imputers, scaling and one-hot encoding are fitted **only on the training data** during model fitting.

This prevents preprocessing information from the test period from leaking into training.


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [
    c for c in X_train.columns
    if c not in numeric_features
]

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ],
    remainder="drop"
)


## 9. Define Regression Models

The project compares multiple regression algorithms rather than assuming one model is best.

Models:
- Linear Regression
- Ridge Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

The regularized Ridge model is included because one-hot encoded categorical features can create a relatively wide feature matrix.


In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(
        alpha=10.0
    ),

    "Decision Tree Regressor": DecisionTreeRegressor(
        max_depth=8,
        random_state=42
    ),

    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=400,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    )
}

model_configs = {
    "Linear Regression": "default",
    "Ridge Regression": "alpha=10.0",
    "Decision Tree Regressor": "max_depth=8",
    "Random Forest Regressor": "n_estimators=400, max_depth=12, min_samples_leaf=2",
    "Gradient Boosting Regressor": "n_estimators=300, learning_rate=0.03, max_depth=3"
}

print("Models:", list(models.keys()))


## 10. Train and Evaluate Every Model

Metrics:
- **MAE:** lower is better.
- **RMSE:** lower is better; penalizes larger errors more.
- **MAPE:** lower is better.
- **R²:** higher is better.

MAPE is calculated safely with a small denominator floor so a zero target cannot cause division-by-zero.


In [ ]:
def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.maximum(np.abs(y_true), 1e-8)
    return np.mean(np.abs((y_true - y_pred) / denominator)) * 100


results = []
trained_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Configuration": model_configs[name],
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "MAPE (%)": safe_mape(y_test, predictions),
        "R2": r2_score(y_test, predictions)
    })

    trained_pipelines[name] = pipeline

results_df = (
    pd.DataFrame(results)
      .sort_values(
          by=["R2", "MAE", "RMSE"],
          ascending=[False, True, True]
      )
      .reset_index(drop=True)
)

display(results_df.round(4))


## 11. Select the Best Model

The ranking prioritizes:
1. Higher R²
2. Lower MAE
3. Lower RMSE

Do not assume Random Forest, Gradient Boosting, or any other model is the winner before running the experiment.


In [ ]:
best_name = results_df.loc[0, "Model"]

print("Best model on the held-out chronological test set:")
print(best_name)

display(results_df.iloc[[0]].round(4))


## 12. Save the Experiment Results

In [ ]:
RESULTS_PATH = "/kaggle/working/experiment_results_table.csv"

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("Saved:", RESULTS_PATH)


## 13. Retrain the Selected Pipeline on All Cleaned Data

After model comparison, the selected preprocessing + model pipeline is fitted on all available cleaned rows.

This final pipeline is the one used for deployment.


In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[best_name])
])

final_model.fit(X, y)

print("Final model:", best_name)
print("Rows used for final training:", len(X))


## 14. Save the Deployment Model

In [ ]:
MODEL_PATH = "/kaggle/working/symca_attendance_model.pkl"

joblib.dump(
    final_model,
    MODEL_PATH
)

print("Saved:", MODEL_PATH)


## 15. Reload and Test the Saved Model

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

sample_prediction = loaded_model.predict(X.iloc[[0]])[0]

print("Sample prediction:", round(sample_prediction, 2), "%")
print("Actual attendance:", round(y.iloc[0], 2), "%")


## 16. Model Performance Summary

In [ ]:
print("Final experiment table:")
display(results_df.round(4))

print("\nSelected model:", best_name)
print("Test MAE:", round(results_df.loc[0, "MAE"], 4))
print("Test RMSE:", round(results_df.loc[0, "RMSE"], 4))
print("Test MAPE:", round(results_df.loc[0, "MAPE (%)"], 4), "%")
print("Test R2:", round(results_df.loc[0, "R2"], 4))


# Final Conclusion

The attendance prediction problem is treated as a **regression** problem because the target `Attendence Percentage` is continuous.

The final workflow:
- Uses the cleaned/engineered dataset from Notebook 2.
- Removes `Students Present` and the target from model inputs.
- Uses a chronological 80/20 split.
- Fits preprocessing only within the training pipeline.
- Compares multiple regression algorithms.
- Reports MAE, RMSE, MAPE and R².
- Selects the best-performing model from the actual test results.
- Saves `experiment_results_table.csv`.
- Saves the complete preprocessing + model pipeline as `symca_attendance_model.pkl`.

**Deployment should use the same feature names and preprocessing structure as this final pipeline.**
